# Notebook 3: Train and compare representations - jumpcp

Per `chemical_surrogate_study/PLAN.md`. 3 targets (`is_active`, `is_active_and_toxic`,
`is_active_not_toxic`) x representations x splits, with 5x5 repeated CV (Guidelines 1,
[Ash2025]) and repeated-measures-ANOVA-based Tukey HSD (Guidelines 2, using the paper's own
reference implementation pattern from `polaris-hub/polaris-method-comparison`, not vanilla
independent-samples Tukey HSD).

In [ ]:
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import pingouin as pg
from rdkit import Chem, DataStructs, rdBase
from rdkit.Chem import Crippen, Descriptors, rdMolDescriptors
from scipy.special import expit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.libqsturng import qsturng, psturng

rdBase.DisableLog("rdApp.*")
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

DATASET_NAME = "jumpcp"
HAS_CLOOME_SPLIT = False
K_MEDOIDS = 190

# Assumes the notebook runs with its own directory (chemical_surrogate_study/notebooks/) as the
# working directory, which is Jupyter's default when opening a notebook.
PROJECT_ROOT = Path.cwd().parent.parent if Path.cwd().name == "notebooks" else Path.cwd()
STUDY_DIR = PROJECT_ROOT / "chemical_surrogate_study"
DATA_DIR = STUDY_DIR / "data"
OLD_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "full_pipeline"  # optional local speedup only, see README
RESULTS_DIR = STUDY_DIR / "results" / DATASET_NAME
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DATASET_PATH = DATA_DIR / "jumpcp_standardized.parquet"

RANDOM_SEED = 42
N_FOLDS = 5
N_REPEATS = 5  # 5x5 repeated CV -> 25 samples, per [Ash2025] Guidelines 1
C_GRID = [0.01, 1.0, 100.0]
TARGETS = ["is_active", "is_active_and_toxic", "is_active_not_toxic"]
N_BOOTSTRAP_OFFICIAL = 25  # matches the 25-sample target from Guidelines 1

torch.manual_seed(RANDOM_SEED)
print(f"Dataset: {DATASET_NAME}  path: {DATASET_PATH}  has_cloome_split: {HAS_CLOOME_SPLIT}  k_medoids: {K_MEDOIDS}")

## Shared helper functions

In [ ]:
def kmedoids_precomputed(D, k, seed=RANDOM_SEED, max_iter=100):
    rng = np.random.default_rng(seed)
    n = D.shape[0]
    medoids = [rng.integers(n)]
    for _ in range(1, k):
        d_to_nearest = D[:, medoids].min(axis=1)
        probs = d_to_nearest ** 2
        probs = probs / probs.sum() if probs.sum() > 0 else np.ones(n) / n
        medoids.append(rng.choice(n, p=probs))
    medoids = np.array(medoids)
    n_iter = 0
    for it in range(max_iter):
        labels = D[:, medoids].argmin(axis=1)
        new_medoids = medoids.copy()
        changed = False
        for c in range(k):
            members = np.flatnonzero(labels == c)
            if len(members) == 0:
                continue
            sub = D[np.ix_(members, members)]
            new_medoid = members[sub.sum(axis=1).argmin()]
            if new_medoid != medoids[c]:
                changed = True
            new_medoids[c] = new_medoid
        medoids = new_medoids
        n_iter = it + 1
        if not changed:
            break
    labels = D[:, medoids].argmin(axis=1)
    return labels, medoids, n_iter


def tanimoto_distance_matrix(fp_bool, chunk=1000):
    X = fp_bool.astype(np.float32)
    counts = X.sum(axis=1)
    n = X.shape[0]
    D = np.empty((n, n), dtype=np.float32)
    for start in range(0, n, chunk):
        end = min(start + chunk, n)
        inter = X[start:end] @ X.T
        union = counts[start:end, None] + counts[None, :] - inter
        sim = np.divide(inter, union, out=np.ones_like(inter), where=union != 0)
        D[start:end] = 1.0 - sim
    return D


def build_group_folds(group_ids, n_folds, seed=RANDOM_SEED):
    counts = pd.Series(group_ids).value_counts()
    order = counts.sample(frac=1, random_state=seed).index
    fold_of_group, fold_sizes = {}, [0] * n_folds
    for group in order:
        size = counts[group]
        target = int(np.argmin(fold_sizes))
        fold_of_group[group] = target
        fold_sizes[target] += size
    return np.array([fold_of_group[g] for g in group_ids])


def physchem_descriptors(smiles, names=[n for n, _ in Descriptors._descList]):
    mol = Chem.MolFromSmiles(smiles) if isinstance(smiles, str) and smiles else None
    if mol is None:
        return np.full(len(names), np.nan, dtype=np.float32)
    try:
        values = Descriptors.CalcMolDescriptors(mol)
        arr = np.array([values.get(name, np.nan) for name in names], dtype=np.float32)
    except Exception:
        arr = np.full(len(names), np.nan, dtype=np.float32)
    return np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)


def logp_only(smiles):
    mol = Chem.MolFromSmiles(smiles) if isinstance(smiles, str) and smiles else None
    if mol is None:
        return np.array([0.0], dtype=np.float32)
    try:
        return np.array([Crippen.MolLogP(mol)], dtype=np.float32)
    except Exception:
        return np.array([0.0], dtype=np.float32)


def morgan_array(smiles_list, n_bits=2048, radius=2, chiral=False):
    arr = np.zeros((len(smiles_list), n_bits), dtype=np.float32)
    for i, s in enumerate(smiles_list):
        mol = Chem.MolFromSmiles(s) if isinstance(s, str) and s else None
        if mol is None:
            continue
        fp = rdMolDescriptors.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits, useChirality=chiral)
        DataStructs.ConvertToNumpyArray(fp, arr[i])
    return arr


def load_cache_by_smiles(cache_path, smiles_list):
    cached = pd.read_parquet(cache_path)
    cached = cached[~cached.index.duplicated(keep="first")]
    return cached.reindex(smiles_list).to_numpy(dtype=np.float32)


# --- CLOOME/CellCLIP: compute embeddings via inference from the public checkpoints, so this
# notebook does not require a precomputed cache to run. ---

CLOOME_REPO_ID = "anasanchezf/cloome"
CLOOME_BIOACTIVITY_CHECKPOINT_FILENAME = "cloome-bioactivity.pt"
CELLCLIP_MODEL_REPO_ID = "suinleelab/CellCLIP"
CELLCLIP_BASE_MODEL = "bert-base-cased"
CELLCLIP_CONTEXT_LENGTH = 256


class CloomeMoleculeMLP(nn.Module):
    """CLOOME's own molecule-encoder architecture: 4-layer MLP over a chiral Morgan fingerprint
    (radius=3, 1024 bits), used here to compute its embeddings via inference from the public
    checkpoint rather than requiring a precomputed cache."""
    def __init__(self, input_dim=1024, hidden_dim=1024, output_dim=512, n_layers=4):
        super().__init__()
        self.layers = nn.ModuleList()
        for layer in range(n_layers):
            dim = input_dim if layer == 0 else hidden_dim
            self.layers.append(nn.Sequential(nn.Linear(dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU()))
        self.layers.append(nn.Sequential(nn.Linear(hidden_dim, output_dim)))

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x


def load_cloome_molecule_encoder(checkpoint_filename=CLOOME_BIOACTIVITY_CHECKPOINT_FILENAME):
    from huggingface_hub import hf_hub_download
    ckpt_path = hf_hub_download(CLOOME_REPO_ID, checkpoint_filename)
    checkpoint = torch.load(ckpt_path, map_location="cpu", weights_only=True)
    prefix = "module.transformer."
    transformer_state_dict = {k[len(prefix):]: v for k, v in checkpoint["state_dict"].items() if k.startswith(prefix)}
    model = CloomeMoleculeMLP()
    missing, unexpected = model.load_state_dict(transformer_state_dict, strict=True)
    assert not missing and not unexpected, f"CLOOME checkpoint mismatch: missing={missing} unexpected={unexpected}"
    model.eval()
    return model


def encode_with_cloome(smiles_list, model, batch_size=512):
    fps = morgan_array(smiles_list, n_bits=1024, radius=3, chiral=True)
    embeddings = []
    with torch.no_grad():
        for start in range(0, len(fps), batch_size):
            batch = torch.from_numpy(fps[start:start + batch_size].astype(np.float32))
            embedding = model(batch)
            embedding = embedding / embedding.norm(dim=-1, keepdim=True)
            embeddings.append(embedding.numpy())
    return np.concatenate(embeddings, axis=0)


class CellClipTextEncoder(nn.Module):
    """CellCLIP's own text-encoder architecture: BERT-base-cased + linear projection to 512-d."""
    def __init__(self):
        super().__init__()
        from transformers import BertModel
        self.text = BertModel.from_pretrained(CELLCLIP_BASE_MODEL)
        self.text_proj = nn.Linear(768, 512)

    def forward(self, input_ids, attention_mask):
        out = self.text(input_ids=input_ids, attention_mask=attention_mask).pooler_output
        return self.text_proj(out)


def load_cellclip_text_encoder():
    from huggingface_hub import hf_hub_download
    from safetensors.torch import load_file as load_safetensors
    ckpt_path = hf_hub_download(CELLCLIP_MODEL_REPO_ID, "model.safetensors")
    full_state_dict = load_safetensors(ckpt_path)
    sub_state_dict = {k: v for k, v in full_state_dict.items() if k.startswith("text.") or k.startswith("text_proj")}
    model = CellClipTextEncoder()
    missing, unexpected = model.load_state_dict(sub_state_dict, strict=True)
    assert not missing and not unexpected, f"CellCLIP checkpoint mismatch: missing={missing} unexpected={unexpected}"
    model.eval()
    return model


def encode_with_cellclip(smiles_list, model, batch_size=64):
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(CELLCLIP_BASE_MODEL)
    prompts = [f"SMILES: {s}" for s in smiles_list]
    tokens = tokenizer(prompts, padding="max_length", truncation=True, max_length=CELLCLIP_CONTEXT_LENGTH, return_tensors="pt")
    embeddings = []
    with torch.no_grad():
        for start in range(0, len(prompts), batch_size):
            batch_ids = tokens["input_ids"][start:start + batch_size]
            batch_mask = tokens["attention_mask"][start:start + batch_size]
            emb = model(batch_ids, batch_mask)
            emb = emb / emb.norm(dim=-1, keepdim=True)
            embeddings.append(emb)
    return torch.cat(embeddings, dim=0).numpy()


def get_or_compute_embeddings(smiles_list, cache_path, encode_fn, legacy_cache_paths=()):
    """SMILES-indexed embedding cache at `cache_path`; computes and appends embeddings for any
    missing SMILES via `encode_fn(missing_smiles) -> array` (runs the encoder's own inference),
    so this works standalone with no pre-existing cache at all. `legacy_cache_paths` are checked
    first, purely as an optional local speedup if a precomputed cache happens to already exist
    there (see README)."""
    if cache_path.exists():
        cached = pd.read_parquet(cache_path)
    else:
        cached = pd.DataFrame()
        for legacy in legacy_cache_paths:
            if legacy.exists():
                cached = pd.read_parquet(legacy)
                cached = cached[~cached.index.duplicated(keep="first")]
                break
    unique_smiles = list(dict.fromkeys(smiles_list))
    missing = [s for s in unique_smiles if s not in cached.index]
    if missing:
        print(f"Computing embeddings for {len(missing):,}/{len(unique_smiles):,} SMILES not in {cache_path.name}...")
        new_emb = encode_fn(missing)
        new_df = pd.DataFrame(new_emb, index=pd.Index(missing, name="__id__"))
        cached = pd.concat([cached, new_df], axis=0)
        cached = cached[~cached.index.duplicated(keep="first")]
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        cached.to_parquet(cache_path)
    return cached.reindex(smiles_list).to_numpy(dtype=np.float32)


def fit_at_c(X_train, y_train, c, seed=RANDOM_SEED):
    clf = Pipeline([("scaler", StandardScaler()),
                     ("clf", LogisticRegression(C=c, max_iter=1000, solver="lbfgs",
                                                 class_weight="balanced", random_state=seed))])
    clf.fit(X_train, y_train)
    return clf


def select_c_single_split(X_train, y_train, seed=RANDOM_SEED, C_grid=C_GRID):
    try:
        Xf, Xv, yf, yv = train_test_split(X_train, y_train, test_size=0.2, random_state=seed, stratify=y_train)
    except ValueError:
        Xf, Xv, yf, yv = train_test_split(X_train, y_train, test_size=0.2, random_state=seed)
    if len(set(yv)) < 2 or len(set(yf)) < 2:
        return C_grid[len(C_grid) // 2]
    best_c, best_score = C_grid[0], -np.inf
    for c in C_grid:
        clf = fit_at_c(Xf, yf, c, seed)
        score = roc_auc_score(yv, clf.predict_proba(Xv)[:, 1])
        if score > best_score:
            best_score, best_c = score, c
    return best_c


def select_c(X_train, y_train, X_val, y_val, C_grid=C_GRID, seed=RANDOM_SEED):
    best_c, best_score = C_grid[0], -np.inf
    for c in C_grid:
        clf = fit_at_c(X_train, y_train, c, seed)
        score = roc_auc_score(y_val, clf.predict_proba(X_val)[:, 1])
        if score > best_score:
            best_score, best_c = score, c
    return best_c


def bootstrap_auc_ci(y_true, y_proba, n_boot=1000, seed=RANDOM_SEED):
    rng_local = np.random.default_rng(seed)
    n = len(y_true)
    boots = [roc_auc_score(y_true[idx], y_proba[idx]) for idx in
             (rng_local.integers(0, n, n) for _ in range(n_boot)) if len(set(y_true[idx])) > 1]
    return (np.nan, np.nan, np.nan) if not boots else (float(np.mean(boots)), float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5)))


class SmallMLP(nn.Module):
    """CLOOME-architecture-style: 4 hidden layers, matching CLOOME_MLP_HIDDEN_LAYERS=4. Layer
    widths are our own reasonable tapering choice (not specified beyond layer count in the source)."""
    def __init__(self, in_dim, hidden=(512, 256, 128, 64), dropout=0.4):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers += [nn.Linear(prev, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


def fit_mlp(X_train, y_train, seed=RANDOM_SEED, max_epochs=150, patience=15, batch_size=256,
            lr=1e-3, weight_decay=1e-4, dropout=0.4):
    try:
        Xf, Xv, yf, yv = train_test_split(X_train, y_train, test_size=0.2, random_state=seed, stratify=y_train)
    except ValueError:
        Xf, Xv, yf, yv = train_test_split(X_train, y_train, test_size=0.2, random_state=seed)
    scaler = StandardScaler().fit(Xf)
    Xf_s = scaler.transform(Xf).astype(np.float32)
    Xv_s = scaler.transform(Xv).astype(np.float32)
    pos_weight = torch.tensor([(len(yf) - yf.sum()) / max(yf.sum(), 1)], dtype=torch.float32)
    model = SmallMLP(X_train.shape[1], dropout=dropout)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    Xf_t, yf_t, Xv_t = torch.tensor(Xf_s), torch.tensor(yf, dtype=torch.float32), torch.tensor(Xv_s)
    n = len(Xf_t)
    best_val_auc, best_state, no_improve, ep = -np.inf, None, 0, 0
    rng = np.random.default_rng(seed)
    for ep in range(max_epochs):
        model.train()
        perm = rng.permutation(n)
        for start in range(0, n, batch_size):
            idx = perm[start:start + batch_size]
            opt.zero_grad()
            loss = loss_fn(model(Xf_t[idx]), yf_t[idx])
            loss.backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            val_logits = model(Xv_t).numpy()
        val_auc = roc_auc_score(yv, val_logits) if len(set(yv)) > 1 else 0.5
        if val_auc > best_val_auc:
            best_val_auc, best_state, no_improve = val_auc, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break
    model.load_state_dict(best_state)
    model.eval()
    return model, scaler, best_val_auc, ep + 1


def mlp_predict_proba(model, scaler, X):
    Xs = scaler.transform(X).astype(np.float32)
    with torch.no_grad():
        logits = model(torch.tensor(Xs, dtype=torch.float32)).numpy()
    return expit(logits)


def rm_tukey_hsd(df, metric, group_col, subject_col, alpha=0.05):
    """Repeated-measures ANOVA + Tukey HSD, matching [Ash2025]'s Guidelines 2 (J. Chem. Inf. Model.
    2025, 65, 9398-9411): the RM-ANOVA's error term feeds the Tukey HSD studentized-range test,
    not a naive independent-samples pooled variance, and Cohen's d uses the paper's own unpaired
    pooled-SD formula (Section 3.3.2). Per the paper's explicit recommendation to always check the
    RM-ANOVA's parametric assumptions, Mauchly's test is run on every comparison, and a
    Greenhouse-Geisser correction is applied to the error degrees of freedom whenever sphericity is
    rejected (found to be violated in most of this study's comparisons)."""
    df_means = df.groupby(group_col)[metric].mean()
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=RuntimeWarning)
        aov = pg.rm_anova(dv=metric, within=group_col, subject=subject_col, data=df, detailed=True, correction=True)
    mse = aov.loc[1, "MS"]
    df_resid = aov.loc[1, "DF"]
    sphericity_ok = bool(aov.loc[0, "sphericity"])
    gg_eps = float(aov.loc[0, "eps"])
    if not sphericity_ok:
        df_resid = df_resid * gg_eps
    methods = df_means.index.tolist()
    n_groups = len(methods)
    n_per_group = df[group_col].value_counts().mean()
    tukey_se = np.sqrt(2 * mse / n_per_group)
    q = qsturng(1 - alpha, n_groups, df_resid)
    rows = []
    for i, m1 in enumerate(methods):
        for j, m2 in enumerate(methods):
            if i < j:
                g1 = df.loc[df[group_col] == m1, metric]
                g2 = df.loc[df[group_col] == m2, metric]
                mean_diff = g1.mean() - g2.mean()
                pooled_sd = np.sqrt((g1.var(ddof=1) + g2.var(ddof=1)) / 2)
                cohens_d = mean_diff / pooled_sd if pooled_sd > 0 else np.nan
                sr = abs(mean_diff) / tukey_se
                p_adj = psturng(sr * np.sqrt(2), n_groups, df_resid)
                p_adj = float(p_adj[0]) if isinstance(p_adj, np.ndarray) else float(p_adj)
                lower = mean_diff - (q / np.sqrt(2) * tukey_se)
                upper = mean_diff + (q / np.sqrt(2) * tukey_se)
                rows.append(dict(group1=m1, group2=m2, meandiff=mean_diff, lower=lower, upper=upper,
                                  p_adj=p_adj, reject=bool(p_adj < alpha), cohens_d=cohens_d,
                                  sphericity=sphericity_ok, gg_epsilon=gg_eps, df_resid_used=df_resid))
    return pd.DataFrame(rows)

## 1. Load standardized dataset, build representations

In [ ]:
pop = pd.read_parquet(DATASET_PATH)
morph_cols = [c for c in pop.columns if c.startswith("morph_")]
print(f"n={len(pop):,}, morphology dims={len(morph_cols)}")
for t in TARGETS:
    print(f"  {t}: {pop[t].sum():,} positive ({pop[t].mean():.1%})")

smiles = pop["SMILES"].tolist()
morph_raw = np.nan_to_num(pop[morph_cols].to_numpy(dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)
physchem_raw = np.nan_to_num(np.stack([physchem_descriptors(s) for s in smiles]), nan=0.0, posinf=0.0, neginf=0.0)
logp_raw = np.nan_to_num(np.stack([logp_only(s) for s in smiles]), nan=0.0, posinf=0.0, neginf=0.0)
morgan_raw = morgan_array(smiles, n_bits=2048, radius=2, chiral=False)
morgan_cloomearch_raw = morgan_array(smiles, n_bits=1024, radius=3, chiral=True)

# CLOOME/CellCLIP embeddings: reuse a local cache under data/ if present, otherwise compute via
# inference from the public checkpoints (downloaded from Hugging Face on first use).
# `OLD_OUTPUT_DIR` caches are only ever an optional local speedup - see README.
cloome_cache_path = DATA_DIR / f"_{DATASET_NAME}_cloome_cache.parquet"
cellclip_cache_path = DATA_DIR / f"_{DATASET_NAME}_cellclip_cache.parquet"

_cloome_model = None
def _cloome_encode_fn(missing_smiles):
    global _cloome_model
    if _cloome_model is None:
        print("Loading CLOOME molecule encoder (anasanchezf/cloome)...")
        _cloome_model = load_cloome_molecule_encoder()
    return encode_with_cloome(missing_smiles, _cloome_model)

_cellclip_model = None
def _cellclip_encode_fn(missing_smiles):
    global _cellclip_model
    if _cellclip_model is None:
        print("Loading CellCLIP text encoder (suinleelab/CellCLIP)...")
        _cellclip_model = load_cellclip_text_encoder()
    return encode_with_cellclip(missing_smiles, _cellclip_model)

cloome_raw = get_or_compute_embeddings(
    smiles, cloome_cache_path, _cloome_encode_fn,
    legacy_cache_paths=[OLD_OUTPUT_DIR / f"{DATASET_NAME}_cloome_bioactivity_cache.parquet"])
cellclip_raw = get_or_compute_embeddings(
    smiles, cellclip_cache_path, _cellclip_encode_fn,
    legacy_cache_paths=[OLD_OUTPUT_DIR / f"{DATASET_NAME}_cellclip_cache.parquet"])
cloome_raw = np.nan_to_num(cloome_raw, nan=0.0, posinf=0.0, neginf=0.0)
cellclip_raw = np.nan_to_num(cellclip_raw, nan=0.0, posinf=0.0, neginf=0.0)

REPR = {"Morphology": morph_raw, "PhysChem": physchem_raw, "LogP": logp_raw, "MorganFP": morgan_raw,
        "CLOOME": cloome_raw, "CellCLIP": cellclip_raw}
print("Linear-probe representations:", {k: v.shape for k, v in REPR.items()})
print("MLP_CLOOME_arch input (radius=3,1024,chiral):", morgan_cloomearch_raw.shape)

y_by_target = {t: pop[t].to_numpy().astype(int) for t in TARGETS}

## 2. Build splits

In [4]:
t0 = time.time()
D = tanimoto_distance_matrix(morgan_raw.astype(bool))
chem_cluster, _, n_iter = kmedoids_precomputed(D, k=K_MEDOIDS, seed=RANDOM_SEED)
print(f"k-medoids (k={K_MEDOIDS}) converged in {n_iter} iterations, {time.time()-t0:.1f}s. "
      f"Cluster sizes: min={np.bincount(chem_cluster).min()}, max={np.bincount(chem_cluster).max()}")
del D

plate_batch = pop["batch_id"].to_numpy()
groups_by_split = {"chemical": chem_cluster, "plate": plate_batch}

if HAS_CLOOME_SPLIT:
    cloome_split = pop["cloome_split"].to_numpy()
    train_mask, val_mask, test_mask = cloome_split == "train", cloome_split == "val", cloome_split == "test"
    print(f"CLOOME official split: train={train_mask.sum()}, val={val_mask.sum()}, test={test_mask.sum()}")

k-medoids (k=190) converged in 4 iterations, 8.2s. Cluster sizes: min=11, max=501


## 3. 5x5 repeated CV (chemical, plate) - linear probes + MLP_CLOOME_arch

In [5]:
rows = []
t_start = time.time()
for split_type, groups in groups_by_split.items():
    for repeat in range(N_REPEATS):
        seed = RANDOM_SEED + repeat
        fold_id = build_group_folds(groups, N_FOLDS, seed=seed)
        for target in TARGETS:
            y = y_by_target[target]
            for fold in range(N_FOLDS):
                test_mask_cv = fold_id == fold
                train_mask_cv = ~test_mask_cv
                y_tr, y_te = y[train_mask_cv], y[test_mask_cv]
                if y_te.sum() < 2 or (len(y_te) - y_te.sum()) < 2 or y_tr.sum() < 2 or (len(y_tr) - y_tr.sum()) < 2:
                    continue
                cv_cycle = f"{repeat}_{fold}"
                for repr_name, X in REPR.items():
                    Xtr, Xte = X[train_mask_cv], X[test_mask_cv]
                    best_c = select_c_single_split(Xtr, y_tr, seed)
                    clf = fit_at_c(Xtr, y_tr, best_c, seed)
                    auc = roc_auc_score(y_te, clf.predict_proba(Xte)[:, 1])
                    rows.append(dict(split_type=split_type, target=target, representation=repr_name,
                                      repeat=repeat, fold=fold, cv_cycle=cv_cycle, auc=auc))
                Xtr_m, Xte_m = morgan_cloomearch_raw[train_mask_cv], morgan_cloomearch_raw[test_mask_cv]
                model, scaler, val_auc, n_epochs = fit_mlp(Xtr_m, y_tr, seed=seed)
                proba = mlp_predict_proba(model, scaler, Xte_m)
                auc = roc_auc_score(y_te, proba)
                rows.append(dict(split_type=split_type, target=target, representation="MLP_CLOOME_arch",
                                  repeat=repeat, fold=fold, cv_cycle=cv_cycle, auc=auc))
            print(f"[{DATASET_NAME}][{split_type}] repeat {repeat+1}/{N_REPEATS} target={target} done, "
                  f"{time.time()-t_start:.0f}s elapsed, {len(rows)} scores so far", flush=True)

cv_scores = pd.DataFrame(rows)
cv_scores.to_csv(RESULTS_DIR / "cv_scores.csv", index=False)
print(f"\nSaved {len(cv_scores):,} CV scores -> {RESULTS_DIR / 'cv_scores.csv'} ({time.time()-t_start:.0f}s total)")
print(cv_scores.groupby(["split_type", "target", "representation"])["auc"].agg(["mean", "std", "count"]).round(3))

[jumpcp][chemical] repeat 1/5 target=is_active done, 174s elapsed, 35 scores so far


[jumpcp][chemical] repeat 1/5 target=is_active_and_toxic done, 368s elapsed, 70 scores so far


[jumpcp][chemical] repeat 1/5 target=is_active_not_toxic done, 528s elapsed, 105 scores so far


[jumpcp][chemical] repeat 2/5 target=is_active done, 712s elapsed, 140 scores so far


[jumpcp][chemical] repeat 2/5 target=is_active_and_toxic done, 898s elapsed, 175 scores so far


...

[jumpcp][plate] repeat 4/5 target=is_active_not_toxic done, 4242s elapsed, 945 scores so far


[jumpcp][plate] repeat 5/5 target=is_active done, 4391s elapsed, 980 scores so far


[jumpcp][plate] repeat 5/5 target=is_active_and_toxic done, 4561s elapsed, 1015 scores so far


[jumpcp][plate] repeat 5/5 target=is_active_not_toxic done, 4697s elapsed, 1050 scores so far



Saved 1,050 CV scores -> /Users/telio/chemical-surrogate-limits/chemical_surrogate_study/results/jumpcp/cv_scores.csv (4697s total)
                                                 mean    std  count
split_type target              representation                      
chemical   is_active           CLOOME           0.663  0.009     25
                               CellCLIP         0.736  0.009     25
                               LogP             0.704  0.012     25
                               MLP_CLOOME_arch  0.740  0.011     25
                               MorganFP         0.752  0.010     25
                               Morphology       0.937  0.004     25
                               PhysChem         0.792  0.008     25
           is_active_and_toxic CLOOME           0.614  0.014     25
                               CellCLIP         0.678  0.008     25
                               LogP             0.615  0.016     25
                               MLP_CLOOME_arch  0.7

## 4. Repeated-measures Tukey HSD per (split, target)

In [6]:
tukey_rows = []
for (split_type, target), sub in cv_scores.groupby(["split_type", "target"]):
    if sub["representation"].nunique() < 2:
        continue
    # ensure a balanced repeated-measures design: keep only cv_cycles present for ALL representations
    counts = sub.groupby("cv_cycle")["representation"].nunique()
    complete_cycles = counts[counts == sub["representation"].nunique()].index
    sub_complete = sub[sub["cv_cycle"].isin(complete_cycles)]
    tab = rm_tukey_hsd(sub_complete, metric="auc", group_col="representation", subject_col="cv_cycle")
    tab["split_type"] = split_type
    tab["target"] = target
    tab["n_cycles"] = len(complete_cycles)
    tukey_rows.append(tab)

tukey_df = pd.concat(tukey_rows, axis=0, ignore_index=True)
tukey_df.to_csv(RESULTS_DIR / "tukey_hsd.csv", index=False)
print(f"Saved Tukey HSD -> {RESULTS_DIR / 'tukey_hsd.csv'} ({len(tukey_df)} pairwise comparisons)")
print(tukey_df[tukey_df["reject"]][["split_type", "target", "group1", "group2", "meandiff", "p_adj", "cohens_d"]].head(20))

Saved Tukey HSD -> /Users/telio/chemical-surrogate-limits/chemical_surrogate_study/results/jumpcp/tukey_hsd.csv (126 pairwise comparisons)
   split_type     target           group1           group2  meandiff  p_adj  \
0    chemical  is_active           CLOOME         CellCLIP -0.073197  0.001   
1    chemical  is_active           CLOOME             LogP -0.040893  0.001   
2    chemical  is_active           CLOOME  MLP_CLOOME_arch -0.077279  0.001   
3    chemical  is_active           CLOOME         MorganFP -0.089149  0.001   
4    chemical  is_active           CLOOME       Morphology -0.274153  0.001   
5    chemical  is_active           CLOOME         PhysChem -0.128969  0.001   
6    chemical  is_active         CellCLIP             LogP  0.032304  0.001   
8    chemical  is_active         CellCLIP         MorganFP -0.015953  0.001   
9    chemical  is_active         CellCLIP       Morphology -0.200956  0.001   
10   chemical  is_active         CellCLIP         PhysChem -0.055773  0

## Done

In [7]:
print("Notebook 3 (jumpcp) complete.")

Notebook 3 (jumpcp) complete.
